In [ ]:
# ================================
# 2. SVM on mobile_price.csv
# ================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score


# ================================
# Load data
# ================================

df = pd.read_csv("mobile_price.csv")

X = df.drop("price_range", axis=1)
y = df["price_range"]


# ================================
# Split data: 60% train, 20% validation, 20% test
# random_state = 42
# ================================

# First split: 60% training, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42,
    shuffle=True,
    stratify=y
)

# Second split: 20% validation, 20% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True,
    stratify=y_temp
)


# ================================
# Feature scaling
# SVM is sensitive to feature scale
# ================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


# ================================
# Helper function
# ================================

def evaluate_svm(model, X_data, y_data, dataset_name):
    y_pred = model.predict(X_data)

    acc = accuracy_score(y_data, y_pred)
    f1 = f1_score(y_data, y_pred, average="macro")

    print(f"{dataset_name} Accuracy: {acc:.4f}")
    print(f"{dataset_name} F1-score: {f1:.4f}")

    return acc, f1

In [ ]:
# ================================
# 2(a) Train SVM with C = 1.0
# ================================

svm_model = SVC(C=1.0)

svm_model.fit(X_train_scaled, y_train)

print("===== SVM Results with C = 1.0 =====")

train_acc, train_f1 = evaluate_svm(
    svm_model,
    X_train_scaled,
    y_train,
    "Training"
)

val_acc, val_f1 = evaluate_svm(
    svm_model,
    X_val_scaled,
    y_val,
    "Validation"
)

test_acc, test_f1 = evaluate_svm(
    svm_model,
    X_test_scaled,
    y_test,
    "Testing"
)

In [ ]:
# ================================
# 2(b) Try different C values
# ================================

C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]

results = []

for C in C_values:
    model = SVC(C=C)
    model.fit(X_train_scaled, y_train)

    y_train_pred = model.predict(X_train_scaled)
    y_val_pred = model.predict(X_val_scaled)
    y_test_pred = model.predict(X_test_scaled)

    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    train_f1 = f1_score(y_train, y_train_pred, average="macro")
    val_f1 = f1_score(y_val, y_val_pred, average="macro")
    test_f1 = f1_score(y_test, y_test_pred, average="macro")

    results.append({
        "C": C,
        "Train Accuracy": train_acc,
        "Validation Accuracy": val_acc,
        "Test Accuracy": test_acc,
        "Train F1": train_f1,
        "Validation F1": val_f1,
        "Test F1": test_f1
    })

results_df = pd.DataFrame(results)

print("===== Results for Different C Values =====")
display(results_df)

In [ ]:
# ================================
# Plot Accuracy
# ================================

plt.figure(figsize=(8, 5))

plt.plot(results_df["C"], results_df["Train Accuracy"], marker="o", label="Training Accuracy")
plt.plot(results_df["C"], results_df["Validation Accuracy"], marker="o", label="Validation Accuracy")
plt.plot(results_df["C"], results_df["Test Accuracy"], marker="o", label="Testing Accuracy")

plt.xscale("log")
plt.xlabel("C value")
plt.ylabel("Accuracy")
plt.title("SVM Accuracy vs C")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ================================
# Plot F1-score
# ================================

plt.figure(figsize=(8, 5))

plt.plot(results_df["C"], results_df["Train F1"], marker="o", label="Training F1")
plt.plot(results_df["C"], results_df["Validation F1"], marker="o", label="Validation F1")
plt.plot(results_df["C"], results_df["Test F1"], marker="o", label="Testing F1")

plt.xscale("log")
plt.xlabel("C value")
plt.ylabel("F1-score")
plt.title("SVM F1-score vs C")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ================================
# Find best C based on validation F1-score
# ================================

best_idx = results_df["Validation F1"].idxmax()
best_result = results_df.loc[best_idx]

print("===== Best C based on Validation F1-score =====")
print(best_result)